In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# A network instead of a table: deep Q-learning

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part A (value-based methods), session 5 (the last method of this part)**
    - values and the Bellman equation, dynamic programming, learning from samples, acting with Q-learning and SARSA, then **a network instead of a table (this session)**
    - learning the policy directly is a separate part that follows
    - **the limitation we start from:** Q-learning keeps one number per state and action; on real problems there are far too many states for a table
    - **the question:** what replaces the table, and what new problems does the replacement bring?
- **This session's goal**
    - scale Q-learning to large state spaces with a network, and see where that path stops
- **By the end you can**
    - say what a network adds over a table (it *generalizes*) and what it costs
    - write the deep Q-learning target and loss, and compute them for a mini-batch
    - explain why *replay* is allowed for Q-learning and not for SARSA
    - explain why the target network is lagged
    - name what a value network cannot do, and the next step that addresses it


# Why a table stops working

**Why this section?** Last step learned $\actvalfun$ as a table. The table was the limitation; to replace it we first need to see exactly what fails.

- Frozen Lake: $16$ states $\times$ $4$ actions $=$ $64$ numbers: a table is fine
- Microduck, the walking robot of the tour: the state is $61$ *continuous* numbers (joint angles and speeds)
- **Failure 1: size.** With continuous numbers the states cannot even be counted, let alone given one entry each
- **Failure 2: no sharing.** A table learns each entry alone
    - a state never visited has no value at all
    - two nearly identical states learn nothing from each other
- A **function with parameters** fixes both
    - it has a fixed size, however many states there are
    - it **generalizes**: similar states get similar values, so what is learned at one state helps at its neighbors

<img src="images/table_vs_network.png" width=70%>

- the figure is a schematic of a network for Microduck: $61$ inputs and $14$ outputs
    - a *Q-network* would have one output per discrete action; Microduck's action is $14$ continuous joint commands, which is why this is not simply a Q-network (this matters on the last slide)


## Three tables a network could replace

**Why this section?** A network can replace the table, but which table? Last step's was one of three in the series, and the choice shapes the rest of the course.

| The table being replaced | Replaced by | What it gives | Where |
|:--|:--|:--|:--|
| action values $\actvalfun(\state, \act)$ | $\actvalfun_\theta(\state, \act)$ | the policy by argmax (*value-based*) | **this step** (deep Q-learning) |
| state values $\statevalfun(\state)$ | $\statevalfun_\theta(\state)$ | a *judge* of the policy, not a policy | the policy-gradient part (a "critic", a learned judge of states) |
| the policy itself $\pi(\act \mid \state)$ | $\pi_\theta(\act \mid \state)$ | the policy, with no argmax | the policy-gradient part |

- the network is the same idea each time: **function approximation**, with parameters $\theta$ trained by gradient steps
- what differs is *which quantity* the network represents, and so what it is trained to match
- this step takes the first row: it is the smallest change to what we already have, since Q-learning already learns $\actvalfun$
- the last slide of this session asks what the first row cannot do; the answer is the third row
- none of the three is a *learned model* of the environment, which is a different idea we do not cover here


## The Q-network and what it is trained to match

**Why this section?** A network must be fed the state and trained on something. Q-learning already supplies what to train on: its target.

- **The network** $\actvalfun_\theta$
    - input: the state $\state$ (for Atari: a stack of recent screen frames, read by a convolutional network, a network that looks at small patches of the image)
    - output: one value $\actvalfun_\theta(\state, \act)$ for each action $\act$ at once, so, in this simple form, the actions must be *discrete* (a short list)
- **The training target** is Q-learning's, unchanged
    - $y = \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun_{\theta^-}(\stateseq_{\tt+1}, \act')$, and $y = \rewseq_{\tt+1}$ if the next state is terminal
    - the $\theta^-$ weights are a lagged copy, explained two slides on
- **The loss** is the mean squared error between $y$ and the prediction $\actvalfun_\theta(\stateseq_\tt, \actseq_\tt)$, averaged over a **mini-batch** (a small set of stored transitions)
    - one gradient step on $\theta$ lowers it
    - $y$ is a bootstrapped sample, as in TD(0): biased while the network is wrong, with lower variance than a Monte Carlo return
    - $y$ is treated as a constant in each gradient step: the gradient flows only through the prediction, not through $y$
- In words: Q-learning's table update, "move the entry toward the target", becomes regression: "move the network's output toward the target"
- **What the network costs**
    - a network update moves the output at *every* state, not one entry, so training can become unstable
    - two standard patches follow: replay and a lagged target network


A mini-batch by numbers ($\disc = 0.9$, two transitions)

| | reward | next state | $\max{\act'} \actvalfun_{\theta^-}$ at the next state | target $y$ | prediction $\actvalfun_\theta(\state, \act)$ |
|:--|:-:|:--|:-:|:-:|:-:|
| 1 | $+1$ | terminal | (none) | $1$ | $0.6$ |
| 2 | $0$ | not terminal | $0.8$ | $0 + 0.9 \times 0.8 = 0.72$ | $0.5$ |

- loss $=$ the mean over the batch of $2$ $= \frac{1}{2}\left[(1 - 0.6)^2 + (0.72 - 0.5)^2\right] \approx 0.10$
- one gradient step on $\theta$ lowers this loss, moving the predictions $0.6$ and $0.5$ toward $1$ and $0.72$; $y$ is held fixed
- the predictions are at the stored states and actions; a mini-batch is used for one gradient step


# Problem 1: the data are not independent, and replay fixes it

**Why this section?** Training a network by gradient steps assumes examples that look like independent draws. An agent's experience does not.

- consecutive transitions of one episode are strongly correlated: similar states, one after another
- the data also *drift*: as the policy changes, the states it visits change
- a network trained on the latest few steps swings toward them and forgets older experience (*catastrophic forgetting*: learning the new overwrites what was learned about the old)
- **Replay memory** is the fix
    - store each **transition**, one step $(\state, \act, \rew, \state')$ plus a flag for "terminal", as it happens
    - a fixed-size first-in-first-out store: the oldest transitions are dropped when it is full
    - draw each mini-batch *uniformly at random*, so one batch mixes steps from many episodes and many older policies
- a batch of shuffled old steps is not an independent draw from the current policy, only far less correlated; it is a mixture of older policies, which raises a question (next slide)
- it also reuses each transition many times: better *sample efficiency*, more learning per step in the environment

<img src="images/a5_replay_batch.png" width=85%>


## Why replay is allowed: the payoff of off-policy

**Why this section?** A stored transition was generated by an *older* policy. Last step asked which policy an update learns about; here is why that matters.

- a transition in the memory came from the behavior policy in force when it was stored: not the current one
- so replay is legitimate only if the update does not care which policy produced the data
- **Q-learning** (off-policy)
    - its one-step target $\rewseq + \disc \max{\act'} \actvalfun(\state', \act')$ uses the *best* next action, not the action the old policy took
    - the reward and next state depend only on the environment given $(\state, \act)$, not on who chose $\act$: each stored transition is a valid sample of the optimality backup, and replay is legal
    - this is exact for a table; with a network it is the standard practice, still without a convergence guarantee
    - one condition: the old policies must have *tried* the relevant state-action pairs, since the update only improves $\actvalfun$ where the memory has data
- **SARSA** (on-policy)
    - its target uses the next action $\act'$ chosen by the policy being learned about
    - an old transition holds an $\act'$ chosen by an *older* policy: the target is then about the wrong policy
    - so SARSA cannot reuse old data without a correction (importance weights, a skipped aside of last step; rarely used)

<img src="images/a5_qlearning_vs_sarsa_target.png" width=80%>

- so the definition from last step has a purpose: **off-policy learning lets us reuse data**, and reuse is what makes network training practical
- the same payoff returns in the policy-gradient part, where old data are reused through a *ratio* of probabilities


# Problem 2: the target moves, and the lagged network fixes it

**Why this section?** The target $y$ is computed with a network, and that network is the one being trained. That makes the target move.

- holding $y$ constant *within* one gradient step does not help across steps: if $y$ uses the current weights $\theta$, every step also changes the *next* target
- the network chases its own output: a moving function and a moving target at once
- this can oscillate or diverge: bootstrapped targets, a network, and off-policy data together are known to be unstable
- the fix: a **target network**, a lagged copy $\theta^-$ of the weights
    - it computes $y$ only
    - it is refreshed from the main network every $C$ steps, so between refreshes the target holds still
- the lag *reduces* the chase, it does not remove the instability

<img src="images/a5_target_chase.png" width=85%>

- both fixes are patches around one fact: we train a network on bootstrapped targets, from data the agent itself produces
    - a table update changes only one entry; a network update changes the output at every state at once


## The deep Q-learning loop

**Why this section?** All the pieces fit in one loop: act, store, sample, learn, sometimes sync.

<img src="images/dqn_loop.png" width=90%>

- the main network acts ($\epsilon$-greedy) and is trained; the target network only provides $y$
- every $C$ steps the target network is refreshed with the main weights


**Pseudo-code for deep Q-learning**

- initialize a replay memory $D$ of capacity $N$; a main network $\actvalfun_\theta$ with random weights $\theta$; a target network $\actvalfun_{\theta^-}$ with weights $\theta^- = \theta$
- set the exploration rate $\epsilon$ (annealed from $1$ to a small value), the discount $\disc$, and the learning rate $\alpha$ (for the optimizer)
- for each episode
    - initialize the state $\state$
    - for each step, until $\state$ is terminal
        - **(1) act:** with probability $\epsilon$ choose a random action $\act$; otherwise $\act = \argmax{\act'} \actvalfun_\theta(\state, \act')$; take $\act$, observe $\rew$ and $\state'$
        - **(2) store:** put the transition $(\state, \act, \rew, \state', \text{done})$ in $D$, where *done* says whether $\state'$ is terminal (one step, not an episode; when $D$ is full, the oldest transition is dropped)
        - **(3) learn:** once $D$ holds enough transitions, sample a random mini-batch of transitions $(\state_j, \act_j, \rew_j, \state'_j, \text{done}_j)$ from $D$
            - target: $y_j = \rew_j$ if $\text{done}_j$ (that is, $\state'_j$ is terminal), otherwise $y_j = \rew_j + \disc \max{\act'} \actvalfun_{\theta^-}(\state'_j, \act')$ ($y_j$ is a constant: no gradient through it)
            - loss: the mean over the batch of $\bigl(y_j - \actvalfun_\theta(\state_j, \act_j)\bigr)^2$; take a gradient step on $\theta$
        - **(4) sync:** every $C$ environment steps, $\theta^- \leftarrow \theta$
        - $\state \leftarrow \state'$ (after the four parts)


## Aside: numbers from the original Atari work, and a code example (for your own review)

- replay memory $N = 10^6$ transitions, target network refreshed every $C = 10^4$ steps, $\epsilon$ annealed from $1$ to $0.1$
- (source: Mnih et al., 2015, *Nature*) one convolutional network reached human-level scores on many of 49 Atari games from the screen pixels alone
- these hyperparameters mattered: deep Q-learning is sensitive to them, and failed runs are common
- [Here](RL_OnPolicy_vs_OffPolicy_code_examples.ipynb) is a notebook with Q-learning with and without replay memory, and SARSA


## Where deep Q-learning sits on the threads

**Why this section?** Each method is placed on the same few questions (the *threads*: model, what is learned, on- or off-policy, backup, bias, variance). Here is this method; the full methods table follows this session.

| Thread | Deep Q-learning |
|:--|:--|
| model needed? | no: it learns from sampled transitions |
| what is learned | action values; the policy is read off by argmax |
| on- or off-policy | **off**: replay reuses old data |
| backup | one sampled next state, one step deep (like TD(0)) |
| bias of the target | biased while the network is wrong (bootstrapped) |
| variance of the target | lower than Monte Carlo (one step) |
| data correlation | consecutive transitions are correlated; replay breaks that |
| the table | replaced by a network: it generalizes |


# The limitation of value-based methods

**Why this section?** Everything so far learned *values* and read the policy off them by an argmax. That last step is where the value-based path stops.

- **The argmax needs a short list of actions**
    - the network outputs one value per action; Microduck's action is $14$ continuous joint commands: there is no list to take a maximum over
    - maximizing over a continuous or enormous action set means an optimization at every single step: possible with extra machinery, but costly
- **The greedy policy is deterministic**
    - in a fully observed problem the best policy can be deterministic, so this is no flaw there
    - but when states look identical to the agent (a *partially observable* problem), or an adversary exploits a predictable agent (rock-paper-scissors: a game, not a single-agent problem), the best policy is random
    - a softmax over $\actvalfun$ can randomize, but nothing trains it to be the *right* randomness
    - exploration is added by hand ($\epsilon$-greedy), where a stochastic policy has randomness built in
- **The policy is learned indirectly**
    - the network is trained on a value-prediction error, not on the return we care about; the policy is whatever the argmax of the predictions happens to be
- **The next step**, for these cases, is the third row of the table of three: put the network on the policy itself
    - $\pi_\theta(\act \mid \state)$ outputs a *distribution* over actions (probabilities for a short list, or the mean and spread of a continuous action)
    - train $\theta$ by gradient steps to raise the expected return directly: **policy gradient**, the next part
    - it is the same move as this session, a network for a table, applied to the policy instead of $\actvalfun$
    - it has its own cost: the training signal is noisy (high variance), which we will see is the main difficulty of that part


# What we established, and what it sets up

- **Established**
    - a table fails on size and on sharing; a network of parameters $\theta$ generalizes across states
    - a network can replace three tables: action values (this step), state values, or the policy (next part)
    - deep Q-learning trains $\actvalfun_\theta$ by regression on Q-learning's target
    - **replay** reuses old transitions, and is legal *because* Q-learning is off-policy; SARSA cannot without a correction
    - the **lagged target network** reduces the bootstrapped target's chase of the network
    - the value-based path ends at the argmax: it needs few discrete actions and gives deterministic policies, a limitation in partially observed and adversarial problems
- **Sets up**
    - next part: policy gradient. The network goes on the policy, and the objective is the expected return itself
    - bias and variance, introduced for the targets of Monte Carlo and TD, return there as the central difficulty
    - off-policy reuse returns as the ratio of probabilities in PPO (a policy-gradient method)
    - the table of methods against threads (model needed, on- or off-policy, backup, bias, variance) closes the value-based path
